## Token Intervention with Generation Afterwards 

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

### Set-up

In [2]:
import sys
sys.path.append("../../src")
sys.path.append("src")

import torch
import gc
import pandas as pd
from tqdm import tqdm

import _util
from _intervention import get_label_probability

In [3]:
model_type = "GPT-OSS_stepwise" # GPT-OSS or R1

if "GPT-OSS" in model_type:
    model, tokenizer = _util.load_OSS()
elif "R1" in model_type:
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
prompt_type = "h" # empty or pre_result or pre_sum
if prompt_type:
    prompt_type = "_" + prompt_type

# Load the divided prompts dataset
if 'h' in prompt_type:
    divided_prompts = pd.read_csv(f"data/{model_type}/h_divided_prompts{prompt_type[2:]}.csv")
else:
    divided_prompts = pd.read_csv(f"data/{model_type}/divided_prompts{prompt_type}.csv")
divided_prompts["base_number"] = divided_prompts["base_number"].astype('Int64')
divided_prompts["source_number"] = divided_prompts["source_number"].astype('Int64')
print(f"loaded {len(divided_prompts)} divided prompts")

loaded 3072 divided prompts


In [5]:
def get_generation_clean_prompt(row):
    return row['base_before'] + str(row['base_number'])

In [6]:
def get_generation_intervention_prompt(row):
    return row['base_before'] + str(row['source_number'])

In [ ]:
# Get header of divided prompts dataset
header = list(divided_prompts.columns) + ['chain_prob', 'post_intervention_factual_chain_prob', 'post_intervention_counterfactual_chain_prob']

filepath = _util.create_csv_file(f"experiments/token_intervention/output/{model_type}/post_intervention_chain_probability", f"{prompt_type[1:]}.csv", header, overwrite=False)

batch_size = 24

for i in tqdm(range(0, len(divided_prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = divided_prompts.iloc[i:i+batch_size]
    
    # Prepare batch of intervention prompts
    clean_prompts = [get_generation_clean_prompt(row) for _, row in batch_rows.iterrows()]
    intervention_prompts = [get_generation_intervention_prompt(row) for _, row in batch_rows.iterrows()]
    factual_suffix = [row['base_after'] for _, row in batch_rows.iterrows()]
    counterfactual_suffix = [row['source_after'] for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    clean_tokens = tokenizer(clean_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    intervention_tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    factual_suffix_tokens = tokenizer(factual_suffix, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="right").to(model.device)["input_ids"]
    counterfactual_suffix_tokens = tokenizer(counterfactual_suffix, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="right").to(model.device)["input_ids"]
    
    # Get label probabilities
    clean_probs = get_label_probability(model, clean_tokens, factual_suffix_tokens, factual_suffix_tokens != tokenizer.pad_token_id)
    intervention_factual_probs = get_label_probability(model, intervention_tokens, factual_suffix_tokens, factual_suffix_tokens != tokenizer.pad_token_id)
    intervention_counterfactual_probs = get_label_probability(model, intervention_tokens, counterfactual_suffix_tokens, counterfactual_suffix_tokens != tokenizer.pad_token_id)
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        _util.write_to_csv(filepath, row.to_list() + [clean_probs[j].item(), intervention_factual_probs[j].item(), intervention_counterfactual_probs[j].item()])


  0%|                                                                                                                       | 0/128 [00:00<?, ?it/s]

 20%|█████████████████████▍                                                                                        | 25/128 [09:35<39:34, 23.06s/it]